# Group Petter

# Team Member Contribution Table

| Member      | Contribution |
| ----------- | ----------- |
| Petter      | Worked on configuring the FastAPI application to load models remotely from the MLFlow server. Documented code. Implemented error handling. |
| Kim        | Worked on major ML issues. See Ensemble - WeakestLink Classification below. |
| Elliot (Scrum Master)     | Worked on major ML issues. See Bootstrap Implementation below. |
| Agust       | Implemented the Stacking and stacking prediction issue by creating a simple training and prediction script for both regression and classification. The script trains multiple base models on the same task (regression or classification), then trains one extra model (the meta-model) that takes the base models predictions as input and learns how to combine them into a better final prediction. The creation of base models part should be removed, and instead connected to the other ML issues we worked on this week to get it to work as shown in the lecture on \"Resampling and Ensemble Techniques |



##  Ensemble — WeakestLink Classification

### Setup

**Dataset:** 1,724 samples after merging, filtering singleton classes, and dropping tied labels. 38 features (13 angle deviation + 25 NASM deviation columns), 10 unique WeakestLink classes.

5-fold CV.

**Base classifiers**  each wrapped in a `BaggingClassifier` with 10 bootstrap bags to introduce data-level diversity on top of model-level diversity:

| Model | Configuration |
|-------|---------------|
| Logistic Regression | C=1.0 |
| LDA | LSQR solver, auto shrinkage |
| KNN | k=5 |
| Random Forest | 200 trees |

**Ensemble strategies:** hard voting and soft voting.

### Results

#### Individual bagged models (5-fold CV macro F1)

| Model | Macro F1 | ± std |
|-------|----------|-------|
| Logistic Regression | 0.4885 | 0.0244 |
| LDA | 0.4718 | 0.0378 |
| Random Forest | 0.4645 | 0.0247 |
| KNN | 0.4368 | 0.0447 |

#### Voting ensembles (5-fold CV macro F1)

| Model | Macro F1 | ± std |
|-------|----------|-------|
| **Soft voting** | **0.5171** | 0.0378 |
| Hard voting | 0.5083 | 0.0364 |

Both ensembles outperformed every individual base model. Soft voting is the best configuration from this experiment.

### Champion Comparison (A4 LightGBM vs Soft Voting Ensemble)

The A4 LightGBM champion was cross-validated on the **same folds** to produce paired per-fold macro F1 scores, enabling a paired t-test.

| Model | Macro F1 (CV mean) | ± std |
|-------|--------------------|-------|
| LightGBM Champion (A4) | 0.5125 | 0.0233 |
| Soft Voting Ensemble (A5) | 0.5171 | 0.0378 |

**Paired t-test:** t = −0.363, **p = 0.735**

The numerical difference is 0.0046 in favour of the ensemble, but is far from statistically significant (p >> 0.05). The A4 LightGBM champion is retained.

### Conclusion

The heterogeneous ensemble with soft voting (macro F1 = 0.517) marginally exceeds the A4 LightGBM champion (0.513) on this CV setup, but the difference is not statistically significant (p = 0.735). No champion change is made and no deployment update is required.


## Bootstrap Implementation

The reader is referred to `ml/models/research/A5` to view the files mentioned below.

In `bootstrap_regression.ipynb`, the work focuses on creating many slightly different versions of the same regression training set using bootstrap sampling and then training the same baseline regressor on each resampled dataset. The notebook first defines a small set of shared utilities: a function that generates bootstrap indices by sampling the training rows with replacement, a function that derives an out-of-bag mask (the rows that were not selected into a given bootstrap draw), and a compact set of regression metrics (R², RMSE, MAE). After the dataset is prepared in the same style as earlier assignments, the notebook makes a single fixed train/test split and explicitly performs bootstrap only on the training split so that every run is evaluated on the same untouched test set. The baseline model used for each bootstrap run is a simple and stable pipeline (median imputation followed by a Ridge regressor), chosen because it is fast enough to train repeatedly. The core "done work" is a loop over 25 bootstrap iterations: for each iteration, bootstrap indices are generated from the training split, the model is cloned and fit on the resampled data, and performance is computed on the shared test set. In addition, the notebook computes optional out-of-bag metrics on the out-of-bag subset of the training split (when such samples exist) to provide an internal check of model behavior without touching the test set. All of this is tracked in MLflow: a parent run logs the overall configuration (seeds, number of bootstraps, sizes, model type), each bootstrap iteration is logged as a nested run with its own parameters and metrics, the bootstrap index array for each iteration is saved and logged as an artifact to make the exact resampling reproducible, and the fitted model is logged as well. At the end, the notebook summarizes what happened across bootstraps by computing mean and standard deviation of the test metrics over the 25 runs, and displays a small descriptive summary table of the collected metrics.

In `bootstrap_classification.ipynb`, the same bootstrap idea is applied to classification, but adapted to classification-specific data preparation and metrics. The notebook defines classification metrics including accuracy, balanced accuracy, macro/weighted F1, and log loss (when predicted probabilities are available). It again defines the bootstrap index generator and out-of-bag mask helper. The dataset preparation step builds the classification target (the notebook creates labels for the classification task and merges them with the feature set), then performs a fixed train/test split with stratification to preserve class balance in both splits. The baseline classifier trained on each bootstrap sample is a stable pipeline with median imputation, standard scaling, and logistic regression. As in the regression notebook, the main work is a loop over 25 bootstrap iterations where the training split is resampled with replacement, the classifier is cloned and trained on the resampled data, and predictions are produced for the fixed test set. Test-set metrics are computed every time, and if the pipeline exposes `predict_proba`, the notebook also computes log loss on the test set. Out-of-bag evaluation is again optional and is computed on the out-of-bag portion of the training split to provide additional insight per bootstrap run. MLflow is used in the same way as in regression: a parent run logs configuration details (including things like the number of classes and feature count), each bootstrap iteration is a nested run that logs its metrics, and each run logs both the saved bootstrap indices as an artifact and the trained model. The notebook also logs the feature column list as an artifact so the exact feature set used is recorded. Finally, it computes and logs parent-level mean and standard deviation across the bootstrap runs for the key classification metrics, and prints a summary of the distribution of metrics across bootstraps.

In `A5.ipynb`, bootstrap is used more broadly as the backbone for building ensembles that are meant to challenge the existing champions, and it is applied to both regression and classification within one consolidated workflow. The notebook starts by defining shared bootstrap utilities (bootstrap index sampling and metric functions) and setting up MLflow tracking for a single experiment. For the regression part, it creates a fixed train/test split and then defines a heterogeneous set of regression model pipelines (not just one baseline). During the bootstrap phase, it runs multiple bootstrap iterations and, for each bootstrap sample, trains every regression model type on the same resampled training data. For each bootstrap and model type combination, it logs a nested MLflow run, stores the model, and records metrics on the shared test set. In addition to logging each individual model, it computes an ensemble prediction for each bootstrap iteration by averaging the predictions from the different model types trained on that same bootstrap sample, and logs that as its own nested run. It also goes beyond simple averaging by training a stacking regressor inside each bootstrap iteration (trained on the bootstrap sample) and logging its test performance as another challenger within the same bootstrap context. After all bootstrap iterations finish, it aggregates across all trained base models from all bootstraps and all model types to form a "global mean" ensemble prediction on the test set, and logs those global ensemble metrics too.

For the classification part in `A5.ipynb`, the same structure is repeated: a fixed stratified train/test split is created; multiple classifier types are defined; bootstrap samples are drawn from the training split; each classifier type is trained on each bootstrap sample; and test-set metrics and models are logged to MLflow for every bootstrap/model combination. Instead of averaging raw class labels, the notebook forms per-bootstrap ensembles by averaging predicted probabilities across the different classifier types within a bootstrap iteration and then deriving final labels from the averaged probabilities; those ensemble metrics are logged as well. It also trains a stacking classifier within each bootstrap iteration and logs its performance. After completing all bootstrap iterations, it computes a global probability-mean ensemble across all trained bootstrap models (across bootstraps and classifier types) and evaluates that global ensemble on the test set.

Finally, `A5.ipynb` also uses bootstrap a second time in a different role: after producing predictions for the champion and the challengers, it performs bootstrap resampling over the test set predictions to estimate uncertainty in the performance differences (for example by generating many resampled test sets and observing how metrics like RMSE/MAE or classification scores change relative to the champion). This part is used to support comparisons with confidence intervals and complements the earlier "bootstrap over the training data" that was used to build the ensembles.